# Classical control (no neural network): identifiability of Cf from h-only data
**Purpose.** Answer, independently of any PINN optimisation, three questions on the Phase-0 twin (24×12 grid, 25 h = 2 M2 cycles):
1. Is a **4-parameter zonal model** (Cf_low, Cf_high, boundary offset b0, boundary slope s) recoverable from **h-only** data? (multi-start L-BFGS on the reference solver, exact `jax.grad`)
2. Is the **free 288-cell field** recoverable (Tikhonov, same solver)?
3. How many **information-bearing modes** does the forward Jacobian have (σ of J/σ_noise > 1), h-only vs h+u+v **at equal observation count**?

Everything is repeated at **weak forcing (U0 = 0.15 m/s)** — the Phase-0 setting, where friction is only a few % of u_t — and **strong forcing (U0 = 1.0 m/s)**, closer to Hormuz tidal currents.
Runs on CPU in ~10–15 min (`QUICK=True`) or ~45 min (`QUICK=False`). **Authoring-sandbox status:** logic exercised with a NumPy mock of JAX; the first real JAX execution is yours — run Cell 3 (gate) first.

In [1]:
import os, json, time
import numpy as np
import jax, jax.numpy as jnp
from jax import lax
from scipy.optimize import minimize
jax.config.update("jax_enable_x64", True)

QUICK = True          # True: ~10-15 min CPU | False: more starts / iterations (~45 min)
TEST_MODE = False     # authoring-sandbox smoke only; leave False on Kaggle

CFG = dict(
    N_STEPS=1500, T0=150, T_STRIDE=25,        # 25 h = ~2 M2 cycles; observe after 2.5 h spin-up, every 25 min
    U0_LIST=[0.15, 1.0],
    P_OBS=39,                                  # divisible by 3 -> exact equal-count comparison h(39 pts) vs h+u+v(13 pts)
    NOISE_FRAC=0.03, SEED=7,
    N_STARTS=3 if QUICK else 8,
    ZONAL_MAXITER=60 if QUICK else 150,
    FREE_MAXITER=120 if QUICK else 300,
    FREE_LAMBDA=1e-3,
    CF_SCALE=2.5e-3,
)
if TEST_MODE:
    CFG.update(N_STEPS=300, T0=60, T_STRIDE=40, N_STARTS=1, ZONAL_MAXITER=3, FREE_MAXITER=2, U0_LIST=[1.0])
REGIMES = {"rich": np.arange(CFG["T0"], CFG["N_STEPS"], CFG["T_STRIDE"]),
           "sparse4": np.linspace(CFG["N_STEPS"] // 4, CFG["N_STEPS"] - 1, 4).astype(int)}   # like the Phase-0 PINN twin (4 snapshots)
print("devices:", jax.devices(), "| CFG:", {k: v for k, v in CFG.items()})
print({k: len(v) for k, v in REGIMES.items()}, "snapshots per regime")

devices: [CpuDevice(id=0)] | CFG: {'N_STEPS': 1500, 'T0': 150, 'T_STRIDE': 25, 'U0_LIST': [0.15, 1.0], 'P_OBS': 39, 'NOISE_FRAC': 0.03, 'SEED': 7, 'N_STARTS': 3, 'ZONAL_MAXITER': 60, 'FREE_MAXITER': 120, 'FREE_LAMBDA': 0.001, 'CF_SCALE': 0.0025}
{'rich': 54, 'sparse4': 4} snapshots per regime


In [2]:
# ---- core physics + problem definition (xp-generic: numpy for truth/tests, jax.numpy for inversion) ----
LX, LY = 150e3, 50e3
NX, NY = 24, 12
DX, DY = LX / NX, LY / NY
G, H0, F0, DT = 9.81, 80.0, 6.5e-5, 60.0
OMEGA = 2 * np.pi / (12.42 * 3600.0)
CF_LOW_TRUE, CF_HIGH_TRUE = 1.5e-3, 4.0e-3
B0_TRUE, S_TRUE = 0.5, 0.1          # boundary x/LX = B0 + S*(y/LY - 0.5)  (identical to the phase-0 twin)
XC = (np.arange(NX) + 0.5) * DX / LX     # normalized cell-centre coordinates
YC = (np.arange(NY) + 0.5) * DY / LY

def swe_step(xp, state, t, Cf, U0):
    eta, u, v = state
    vp = xp.pad(v, ((1, 1), (0, 0)), mode="edge"); vc = 0.5 * (vp[:, :-1] + vp[:, 1:]); v_at_u = 0.5 * (vc[:-1, :] + vc[1:, :])
    uc = 0.5 * (u[:-1, :] + u[1:, :]); up = xp.pad(uc, ((0, 0), (1, 1)), mode="edge"); u_at_v = 0.5 * (up[:, :-1] + up[:, 1:])
    Cf_u = Cf[np.clip(np.arange(NX + 1), 0, NX - 1), :]
    Cf_v = Cf[:, np.clip(np.arange(NY + 1), 0, NY - 1)]
    su = xp.sqrt(u ** 2 + v_at_u ** 2 + 1e-8); sv = xp.sqrt(u_at_v ** 2 + v ** 2 + 1e-8)
    de_dx = (eta[1:, :] - eta[:-1, :]) / DX; de_dy = (eta[:, 1:] - eta[:, :-1]) / DY
    fu = Cf_u / H0 * u * su; fv = Cf_v / H0 * v * sv
    un = u[1:-1, :] + DT * (-G * de_dx + F0 * v_at_u[1:-1, :] - fu[1:-1, :])
    u_new = xp.concatenate([xp.full((1, NY), U0 * xp.cos(OMEGA * t)), un, xp.zeros((1, NY))], axis=0)
    vn = v[:, 1:-1] + DT * (-G * de_dy - F0 * u_at_v[:, 1:-1] - fv[:, 1:-1])
    v_new = xp.concatenate([xp.zeros((NX, 1)), vn, xp.zeros((NX, 1))], axis=1)
    div = (u_new[1:, :] - u_new[:-1, :]) / DX + (v_new[:, 1:] - v_new[:, :-1]) / DY
    return (eta - DT * H0 * div, u_new, v_new)

def run_obs_np(Cf, U0, n_steps, ox, oy):
    """numpy reference run -> (h,u,v) at observed cells for every step: 3 arrays (n_steps, P)."""
    s = (np.zeros((NX, NY)), np.zeros((NX + 1, NY)), np.zeros((NX, NY + 1)))
    H, U, V = [], [], []
    for n in range(n_steps):
        s = swe_step(np, s, n * DT, Cf, U0)
        e, u, v = s
        H.append(e[ox, oy]); U.append((0.5 * (u[:-1] + u[1:]))[ox, oy]); V.append((0.5 * (v[:, :-1] + v[:, 1:]))[ox, oy])
    return np.array(H), np.array(U), np.array(V)

def cf_zonal(xp, p, w_cells=0.5):
    """p = [ln Cf_low, ln Cf_high, b0, s]; smooth two-zone field on the cell grid (width = w_cells cells)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    arg = (X - (p[2] + p[3] * (Y - 0.5))) / (w_cells * DX / LX)
    sig = 1.0 / (1.0 + xp.exp(-arg))
    lo, hi = xp.exp(p[0]), xp.exp(p[1])
    return lo + (hi - lo) * sig

def cf_true_field():
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    return np.where(X < B0_TRUE + S_TRUE * (Y - 0.5), CF_LOW_TRUE, CF_HIGH_TRUE)


# ---- JAX forward model returning only the observed cells (keeps reverse/forward-mode memory tiny) ----
def make_forward(U0, ox, oy):
    oxj, oyj = jnp.array(ox), jnp.array(oy)
    def fwd(Cf):
        def body(state, n):
            new = swe_step(jnp, state, n * DT, Cf, U0)
            e, u, v = new
            uc = 0.5 * (u[:-1, :] + u[1:, :]); vc = 0.5 * (v[:, :-1] + v[:, 1:])
            return new, (e[oxj, oyj], uc[oxj, oyj], vc[oxj, oyj])
        init = (jnp.zeros((NX, NY)), jnp.zeros((NX + 1, NY)), jnp.zeros((NX, NY + 1)))
        _, ys = lax.scan(body, init, jnp.arange(CFG["N_STEPS"]))
        return ys
    return fwd

CF_T = cf_true_field()
BOUNDS = [(np.log(5e-4), np.log(2e-2))] * 2 + [(0.1, 0.9), (-0.5, 0.5)]
P_TRUE = np.array([np.log(CF_LOW_TRUE), np.log(CF_HIGH_TRUE), B0_TRUE, S_TRUE])

def build_problem(U0, regime, variant, P_use):
    """Truth is generated by the independent NumPy loop; inversion uses the JAX solver (same equations)."""
    rng = np.random.default_rng(CFG["SEED"]); cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False)
    ox, oy = np.unravel_index(cells[:P_use], (NX, NY))
    t_idx = REGIMES[regime]
    Hc, Uc, Vc = run_obs_np(CF_T, U0, CFG["N_STEPS"], ox, oy)
    clean = dict(h=Hc[t_idx], u=Uc[t_idx], v=Vc[t_idx])
    sd = {k: CFG["NOISE_FRAC"] * clean[k].std() for k in clean}
    nr = np.random.default_rng(11)
    obs = {k: clean[k] + nr.normal(scale=sd[k], size=clean[k].shape) for k in clean}
    fwd = make_forward(U0, ox, oy)
    tj = jnp.array(t_idx)
    def chi2(Cf):
        H, U, V = fwd(Cf)
        c = jnp.mean(((H[tj] - obs["h"]) / sd["h"]) ** 2)
        if variant == "huv":
            c = (c + jnp.mean(((U[tj] - obs["u"]) / sd["u"]) ** 2) + jnp.mean(((V[tj] - obs["v"]) / sd["v"]) ** 2)) / 3.0
        return c
    return dict(U0=U0, regime=regime, variant=variant, P=P_use, ox=ox, oy=oy, t_idx=t_idx, obs=obs, sd=sd,
                fwd=fwd, chi2=chi2, n_obs=int(obs["h"].size * (3 if variant == "huv" else 1)))

## Cell 3 — GATE: solver cross-check and gradient check (must pass before anything else)

In [3]:
gate = {}
pb = build_problem(CFG["U0_LIST"][-1], "rich" if not TEST_MODE else "rich", "h", CFG["P_OBS"])
# (a) JAX solver == NumPy solver (independent implementations of the same step)
H_j, U_j, V_j = [np.asarray(a) for a in pb["fwd"](jnp.array(CF_T))]
H_n, U_n, V_n = run_obs_np(CF_T, pb["U0"], CFG["N_STEPS"], pb["ox"], pb["oy"])
rel = np.max(np.abs(H_j - H_n)) / np.std(H_n)
print(f"(a) JAX vs NumPy solver: max|dh|/std(h) = {rel:.2e}  ->", "PASS" if rel < 1e-8 else "FAIL")
gate["solver_match"] = bool(rel < 1e-8)
# (b) jax.grad vs central finite differences on the 4 zonal parameters
loss_p = lambda p: pb["chi2"](cf_zonal(jnp, p))
vg = jax.jit(jax.value_and_grad(loss_p))
p0 = P_TRUE + np.array([0.2, -0.2, 0.03, -0.02])
v0, g0 = vg(jnp.array(p0)); g0 = np.asarray(g0)
fd = np.zeros(4)
for i in range(4):
    e = np.zeros(4); e[i] = 1e-6
    fd[i] = (float(loss_p(jnp.array(p0 + e))) - float(loss_p(jnp.array(p0 - e)))) / 2e-6
err = np.max(np.abs(g0 - fd) / (np.abs(fd) + 1e-12))
print(f"(b) grad check: max rel err = {err:.2e}  (jax {np.round(g0,4)} | fd {np.round(fd,4)}) ->", "PASS" if err < 1e-3 else "FAIL")
gate["grad_ok"] = bool(err < 1e-3)
# (c) chi2 at the true parameters is ~1 (noise-level fit) — sanity of the noise model
print("(c) chi2 at true params (h-only, U0=%.2f): %.3f (expect ~1.0)" % (pb["U0"], float(loss_p(jnp.array(P_TRUE)))))
assert all(gate.values()), "GATE FAILED — do not interpret anything below; paste this cell's output to the assistant."

(a) JAX vs NumPy solver: max|dh|/std(h) = 8.07e-15  -> PASS
(b) grad check: max rel err = 7.98e-09  (jax [-1.0159 -1.1263  2.2813 -0.0274] | fd [-1.0159 -1.1263  2.2813 -0.0274]) -> PASS
(c) chi2 at true params (h-only, U0=1.00): 0.999 (expect ~1.0)


## Cell 4 — Zonal 4-parameter model, multi-start (h-only vs h+u+v at equal observation count)

In [4]:
def fit_zonal(pb, p0):
    vgf = jax.jit(jax.value_and_grad(lambda p: pb["chi2"](cf_zonal(jnp, p))))
    def f(p):
        v, g = vgf(jnp.array(p)); return float(v), np.asarray(g, dtype=np.float64)
    res = minimize(f, p0, jac=True, method="L-BFGS-B", bounds=BOUNDS, options=dict(maxiter=CFG["ZONAL_MAXITER"]))
    p = res.x
    bnd_km = float(np.mean(np.abs((p[2] - B0_TRUE) + (p[3] - S_TRUE) * (YC - 0.5))) * LX / 1e3)
    return dict(cf_low=float(np.exp(p[0])), cf_high=float(np.exp(p[1])), b0=float(p[2]), s=float(p[3]),
                chi2=float(res.fun), nit=int(res.nit), bnd_km=bnd_km)

RESULTS = {"zonal": [], "free": [], "jac": [], "cfg": {k: (v if not isinstance(v, np.ndarray) else v.tolist()) for k, v in CFG.items()}}
ZON_CONFIGS = [("h", CFG["P_OBS"]), ("huv", CFG["P_OBS"] // 3)]       # equal observation count (39 vs 3*13)
t0 = time.time()
for U0 in CFG["U0_LIST"]:
    for regime in REGIMES:
        for variant, P_use in ZON_CONFIGS:
            pbz = build_problem(U0, regime, variant, P_use)
            sols = []
            for k in range(CFG["N_STARTS"]):
                r = np.random.default_rng(100 + k)
                p0 = np.array([np.log(2.5e-3) + r.normal(0, 0.4), np.log(2.5e-3) + r.normal(0, 0.4), r.uniform(0.3, 0.7), r.uniform(-0.2, 0.2)])
                sols.append(fit_zonal(pbz, p0))
            best = min(s["chi2"] for s in sols)
            equiv = [s for s in sols if s["chi2"] <= best + 0.02]            # solutions statistically as good as the best
            spread = lambda key: (min(s[key] for s in equiv), max(s[key] for s in equiv))
            rec = dict(U0=U0, regime=regime, variant=variant, n_obs=pbz["n_obs"], best_chi2=best, n_starts=len(sols),
                       n_equiv=len(equiv), bnd_km_best=min(s["bnd_km"] for s in equiv), bnd_km_worst=max(s["bnd_km"] for s in equiv),
                       cf_low_range=spread("cf_low"), cf_high_range=spread("cf_high"), b0_range=spread("b0"), sols=sols)
            RESULTS["zonal"].append(rec)
            print(f"[zonal] U0={U0:<4} {regime:8s} {variant:3s} nobs={pbz['n_obs']:5d} | best chi2={best:.3f} | {len(equiv)}/{len(sols)} starts equivalent | "
                  f"boundary err {rec['bnd_km_best']:.1f}-{rec['bnd_km_worst']:.1f} km | Cf_low {rec['cf_low_range'][0]:.5f}-{rec['cf_low_range'][1]:.5f} "
                  f"Cf_high {rec['cf_high_range'][0]:.5f}-{rec['cf_high_range'][1]:.5f}  [{time.time()-t0:.0f}s]")
print("truth: Cf_low=%.5f Cf_high=%.5f b0=%.2f s=%.2f" % (CF_LOW_TRUE, CF_HIGH_TRUE, B0_TRUE, S_TRUE))

[zonal] U0=0.15 rich     h   nobs= 2106 | best chi2=0.995 | 3/3 starts equivalent | boundary err 13.0-60.0 km | Cf_low 0.00249-0.00310 Cf_high 0.00114-0.00274  [16s]
[zonal] U0=0.15 rich     huv nobs= 2106 | best chi2=0.997 | 2/3 starts equivalent | boundary err 1.6-23.7 km | Cf_low 0.00186-0.00209 Cf_high 0.00289-0.00350  [32s]
[zonal] U0=0.15 sparse4  h   nobs=  156 | best chi2=0.867 | 3/3 starts equivalent | boundary err 21.4-56.8 km | Cf_low 0.00155-0.00275 Cf_high 0.00050-0.00397  [49s]
[zonal] U0=0.15 sparse4  huv nobs=  156 | best chi2=0.866 | 3/3 starts equivalent | boundary err 10.9-28.5 km | Cf_low 0.00190-0.00287 Cf_high 0.00164-0.00330  [63s]
[zonal] U0=1.0  rich     h   nobs= 2106 | best chi2=0.994 | 2/3 starts equivalent | boundary err 5.7-5.7 km | Cf_low 0.00174-0.00174 Cf_high 0.00356-0.00356  [77s]
[zonal] U0=1.0  rich     huv nobs= 2106 | best chi2=2.036 | 1/3 starts equivalent | boundary err 16.0-16.0 km | Cf_low 0.00276-0.00276 Cf_high 0.00200-0.00200  [90s]
[zonal]

## Cell 5 — Free 288-cell field (Tikhonov on ln Cf), same solver

In [5]:
def fit_free(pb, lam):
    chi2 = pb["chi2"]
    def obj(q):
        lnc = q.reshape(NX, NY)
        smooth = jnp.mean((lnc[1:, :] - lnc[:-1, :]) ** 2) + jnp.mean((lnc[:, 1:] - lnc[:, :-1]) ** 2)
        return chi2(jnp.exp(lnc)) + lam * smooth
    vgf = jax.jit(jax.value_and_grad(obj))
    def f(q):
        v, g = vgf(jnp.array(q)); return float(v), np.asarray(g, dtype=np.float64)
    q0 = np.full(NX * NY, np.log(2.5e-3))
    res = minimize(f, q0, jac=True, method="L-BFGS-B", bounds=[(np.log(5e-4), np.log(2e-2))] * (NX * NY), options=dict(maxiter=CFG["FREE_MAXITER"]))
    cf = np.exp(res.x).reshape(NX, NY)
    r = float(np.corrcoef(cf.ravel(), CF_T.ravel())[0, 1]); rng_ = CF_T.max() - CF_T.min()
    rmse = float(100 * np.sqrt(np.mean((cf - CF_T) ** 2)) / rng_)
    return dict(r=r, rmse_pct=rmse, obj=float(res.fun), nit=int(res.nit), cf_std=float(cf.std()))

t0 = time.time()
for U0 in CFG["U0_LIST"]:
    for regime in REGIMES:
        for variant, P_use in ZON_CONFIGS:
            pbf = build_problem(U0, regime, variant, P_use)
            out = fit_free(pbf, CFG["FREE_LAMBDA"]); out.update(U0=U0, regime=regime, variant=variant, n_obs=pbf["n_obs"])
            RESULTS["free"].append(out)
            print(f"[free ] U0={U0:<4} {regime:8s} {variant:3s} nobs={pbf['n_obs']:5d} | r(Cf)={out['r']:.3f}  RMSE={out['rmse_pct']:.1f}%  std(Cf_hat)={out['cf_std']:.5f}  nit={out['nit']}  [{time.time()-t0:.0f}s]")

[free ] U0=0.15 rich     h   nobs= 2106 | r(Cf)=-0.027  RMSE=208.8%  std(Cf_hat)=0.00503  nit=104  [16s]
[free ] U0=0.15 rich     huv nobs= 2106 | r(Cf)=0.255  RMSE=94.3%  std(Cf_hat)=0.00234  nit=120  [34s]
[free ] U0=0.15 sparse4  h   nobs=  156 | r(Cf)=0.022  RMSE=257.5%  std(Cf_hat)=0.00630  nit=120  [52s]
[free ] U0=0.15 sparse4  huv nobs=  156 | r(Cf)=0.179  RMSE=179.9%  std(Cf_hat)=0.00454  nit=120  [70s]
[free ] U0=1.0  rich     h   nobs= 2106 | r(Cf)=0.262  RMSE=138.4%  std(Cf_hat)=0.00356  nit=120  [88s]
[free ] U0=1.0  rich     huv nobs= 2106 | r(Cf)=0.824  RMSE=28.9%  std(Cf_hat)=0.00095  nit=120  [106s]
[free ] U0=1.0  sparse4  h   nobs=  156 | r(Cf)=0.275  RMSE=144.1%  std(Cf_hat)=0.00373  nit=120  [124s]
[free ] U0=1.0  sparse4  huv nobs=  156 | r(Cf)=0.519  RMSE=67.4%  std(Cf_hat)=0.00195  nit=120  [142s]


## Cell 6 — Information content: singular values of the noise-normalised forward Jacobian (equal observation count)

In [6]:
def jacobian_snr(pb, chunk=36):
    t_j = jnp.array(pb["t_idx"]); sd = pb["sd"]; fwd = pb["fwd"]; variant = pb["variant"]
    def f(x):
        H, U, V = fwd((x * CFG["CF_SCALE"]).reshape(NX, NY))
        out = [(H[t_j] / sd["h"]).ravel()]
        if variant == "huv":
            out += [(U[t_j] / sd["u"]).ravel(), (V[t_j] / sd["v"]).ravel()]
        return jnp.concatenate(out)
    x0 = jnp.array(CF_T.ravel() / CFG["CF_SCALE"])
    jvp_one = jax.jit(jax.vmap(lambda tv: jax.jvp(f, (x0,), (tv,))[1]))
    eye = jnp.eye(NX * NY)
    cols = [np.asarray(jvp_one(eye[i:i + chunk])) for i in range(0, NX * NY, chunk)]
    J = np.concatenate(cols, axis=0).T              # (n_obs, 288)
    s = np.linalg.svd(J, compute_uv=False)
    return s

t0 = time.time()
for U0 in CFG["U0_LIST"]:
    for variant, P_use in [("h", CFG["P_OBS"]), ("huv", CFG["P_OBS"] // 3), ("huv", CFG["P_OBS"])]:
        pbj = build_problem(U0, "rich", variant, P_use)
        s = jacobian_snr(pbj)
        rec = dict(U0=U0, variant=variant, P=P_use, n_obs=pbj["n_obs"], n_modes_snr_gt_1=int((s > 1).sum()), n_modes_snr_gt_10=int((s > 10).sum()),
                   rank_rel_1e6=int((s > s[0] * 1e-6).sum()), cond=float(s[0] / max(s[-1], 1e-300)), sv_top=float(s[0]))
        RESULTS["jac"].append(rec)
        print(f"[jac  ] U0={U0:<4} {variant:3s} P={P_use:2d} nobs={pbj['n_obs']:5d} | modes with sigma>1 (SNR>1): {rec['n_modes_snr_gt_1']:3d}/288 | >10: {rec['n_modes_snr_gt_10']:3d} | rank(1e-6)={rec['rank_rel_1e6']:3d} | cond={rec['cond']:.1e}  [{time.time()-t0:.0f}s]")

[jac  ] U0=0.15 h   P=39 nobs= 2106 | modes with sigma>1 (SNR>1):   2/288 | >10:   0 | rank(1e-6)=287 | cond=3.6e+300  [7s]
[jac  ] U0=0.15 huv P=13 nobs= 2106 | modes with sigma>1 (SNR>1):  48/288 | >10:  12 | rank(1e-6)=287 | cond=3.9e+301  [15s]
[jac  ] U0=0.15 huv P=39 nobs= 6318 | modes with sigma>1 (SNR>1): 110/288 | >10:  36 | rank(1e-6)=287 | cond=3.6e+301  [22s]
[jac  ] U0=1.0  h   P=39 nobs= 2106 | modes with sigma>1 (SNR>1):  10/288 | >10:   1 | rank(1e-6)=287 | cond=1.1e+301  [30s]
[jac  ] U0=1.0  huv P=13 nobs= 2106 | modes with sigma>1 (SNR>1): 133/288 | >10:  32 | rank(1e-6)=287 | cond=3.7e+302  [37s]
[jac  ] U0=1.0  huv P=39 nobs= 6318 | modes with sigma>1 (SNR>1): 206/288 | >10:  83 | rank(1e-6)=287 | cond=3.3e+302  [45s]


## Cell 7 — Save and print the one-screen summary (paste this output back)

In [7]:
out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
path = os.path.join(out_dir, "control_results.json")
json.dump(RESULTS, open(path, "w"), indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
print("saved ->", path)
print("\n=== SUMMARY (zonal 4-param, h-only vs h+u+v, equal obs count) ===")
for r in RESULTS["zonal"]:
    print(f"U0={r['U0']:<4} {r['regime']:8s} {r['variant']:3s} | equiv starts {r['n_equiv']}/{r['n_starts']} | boundary err {r['bnd_km_best']:.1f}-{r['bnd_km_worst']:.1f} km | best chi2 {r['best_chi2']:.3f}")
print("\n=== SUMMARY (free field) ===")
for r in RESULTS["free"]:
    print(f"U0={r['U0']:<4} {r['regime']:8s} {r['variant']:3s} | r={r['r']:.3f}  RMSE={r['rmse_pct']:.1f}%")
print("\n=== SUMMARY (Jacobian information modes) ===")
for r in RESULTS["jac"]:
    print(f"U0={r['U0']:<4} {r['variant']:3s} nobs={r['n_obs']:5d} | SNR>1 modes: {r['n_modes_snr_gt_1']:3d}/288")

saved -> /kaggle/working/control_results.json

=== SUMMARY (zonal 4-param, h-only vs h+u+v, equal obs count) ===
U0=0.15 rich     h   | equiv starts 3/3 | boundary err 13.0-60.0 km | best chi2 0.995
U0=0.15 rich     huv | equiv starts 2/3 | boundary err 1.6-23.7 km | best chi2 0.997
U0=0.15 sparse4  h   | equiv starts 3/3 | boundary err 21.4-56.8 km | best chi2 0.867
U0=0.15 sparse4  huv | equiv starts 3/3 | boundary err 10.9-28.5 km | best chi2 0.866
U0=1.0  rich     h   | equiv starts 2/3 | boundary err 5.7-5.7 km | best chi2 0.994
U0=1.0  rich     huv | equiv starts 1/3 | boundary err 16.0-16.0 km | best chi2 2.036
U0=1.0  sparse4  h   | equiv starts 2/3 | boundary err 0.6-0.6 km | best chi2 0.885
U0=1.0  sparse4  huv | equiv starts 2/3 | boundary err 1.5-1.5 km | best chi2 0.919

=== SUMMARY (free field) ===
U0=0.15 rich     h   | r=-0.027  RMSE=208.8%
U0=0.15 rich     huv | r=0.255  RMSE=94.3%
U0=0.15 sparse4  h   | r=0.022  RMSE=257.5%
U0=0.15 sparse4  huv | r=0.179  RMSE=179.9%
